# Projet 1 — Diagnostic médical assisté : pipeline de données multimodal
**Dataset :** NIH Chest X-ray14 (`nih-chest-xrays/data`) — radiographies thoraciques + métadonnées cliniques.

Ce notebook implémente tout le pipeline DataOps avant l'entraînement :

1. Ingestion & inventaire des fichiers
2. Validation de schéma & qualité
3. **Anonymisation** (pseudonymisation des patients)
4. **Alignement image ↔ dossier patient**
5. Nettoyage (aberrants, doublons, cohérence)
6. EDA
7. Feature engineering (labels multi-hot, historique patient)
8. **Valeurs manquantes** (imputation + indicateurs)
9. Split train/val/test **par patient** (sans fuite)
10. Prétraitement tabulaire (sklearn) & image (CLAHE, augmentation)
11. Dataset / DataLoader PyTorch multimodal + export des artefacts

> Sur Kaggle : *Add Input* → `nih-chest-xrays/data`, accélérateur GPU recommandé.

In [ ]:
!pip install -q albumentations iterative-stratification pandera missingno

In [ ]:
import os, glob, hashlib, json, warnings, secrets
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import cv2
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import joblib
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

SEED = 42
np.random.seed(SEED)
# Détection automatique du dossier (Kaggle a changé ses chemins : /kaggle/input/data
# ou /kaggle/input/datasets/nih-chest-xrays/data selon la version)
_csv = glob.glob('/kaggle/input/**/Data_Entry_2017.csv', recursive=True)
assert _csv, 'Dataset NIH introuvable : ajoutez nih-chest-xrays/data via Add Input'
DATA_DIR = os.path.dirname(_csv[0])
print('DATA_DIR =', DATA_DIR)
OUT_DIR = '/kaggle/working/artifacts'
os.makedirs(OUT_DIR, exist_ok=True)
IMG_SIZE = 224
SAMPLE_FRAC = 1.0   # mettre 0.1 pour itérer vite

## 1. Ingestion & inventaire des images

In [ ]:
meta_raw = pd.read_csv(f'{DATA_DIR}/Data_Entry_2017.csv')
print(meta_raw.shape)
meta_raw.head()

In [ ]:
# Les 112 120 images sont réparties dans images_001 ... images_012
img_paths = glob.glob(f'{DATA_DIR}/images_*/images/*.png')
path_map = {os.path.basename(p): p for p in img_paths}
print('Images trouvées sur disque :', len(path_map))

with open(f'{DATA_DIR}/train_val_list.txt') as f: official_trainval = set(f.read().split())
with open(f'{DATA_DIR}/test_list.txt') as f: official_test = set(f.read().split())
print('train_val officiel :', len(official_trainval), '| test officiel :', len(official_test))

## 2. Validation de schéma & qualité

In [ ]:
meta = meta_raw.copy()
# Colonnes inutiles ou vides
meta = meta.loc[:, ~meta.columns.str.startswith('Unnamed')]
meta = meta.rename(columns={
    'Image Index': 'image_id', 'Finding Labels': 'labels', 'Follow-up #': 'follow_up',
    'Patient ID': 'patient_id', 'Patient Age': 'age', 'Patient Gender': 'sex',
    'View Position': 'view', 'OriginalImage[Width': 'orig_w', 'Height]': 'orig_h',
    'OriginalImagePixelSpacing[x': 'spacing_x', 'y]': 'spacing_y'})
print(meta.dtypes)

report = {
    'lignes': len(meta),
    'image_id dupliqués': int(meta.image_id.duplicated().sum()),
    'lignes dupliquées': int(meta.duplicated().sum()),
    'manquants par colonne': meta.isna().sum().to_dict(),
    'valeurs sexe': meta.sex.value_counts().to_dict(),
    'valeurs vue': meta.view.value_counts().to_dict(),
    'âge min/max': (int(meta.age.min()), int(meta.age.max())),
    'âges > 100': int((meta.age > 100).sum()),
}
for k, v in report.items(): print(f'{k}: {v}')

In [ ]:
# Contrat de données avec pandera (avant nettoyage on attend des écarts, on les liste)
import pandera as pa
schema = pa.DataFrameSchema({
    'image_id': pa.Column(str, pa.Check.str_matches(r'^\d{8}_\d{3}\.png$'), unique=True),
    'patient_id': pa.Column(int, pa.Check.ge(1)),
    'age': pa.Column(float, pa.Check.in_range(0, 100), nullable=True, coerce=True),
    'sex': pa.Column(str, pa.Check.isin(['M', 'F'])),
    'view': pa.Column(str, pa.Check.isin(['PA', 'AP'])),
    'follow_up': pa.Column(int, pa.Check.ge(0)),
})
try:
    schema.validate(meta, lazy=True)
    print('Schéma OK')
except pa.errors.SchemaErrors as e:
    print('Violations détectées :')
    display(e.failure_cases.groupby(['column', 'check']).size())

## 3. Anonymisation
Les données NIH sont déjà dé-identifiées, mais on applique le même traitement que sur des données hospitalières réelles :
- **pseudonymisation irréversible** de `patient_id` (SHA-256 + sel secret, jamais versionné) ;
- suppression des métadonnées techniques non nécessaires pouvant servir à la ré-identification (dimensions originales, pixel spacing) ;
- âges > 89 regroupés (règle HIPAA *Safe Harbor*) ;
- les noms de fichiers contiennent l'ID patient → on crée un `image_key` neutre et on garde la table de correspondance hors des données de modélisation.

In [ ]:
SALT = os.environ.get('PSEUDO_SALT') or secrets.token_hex(16)  # en prod : coffre-fort de secrets
def pseudo(x, n=16): return hashlib.sha256(f'{SALT}{x}'.encode()).hexdigest()[:n]

meta['patient_key'] = meta.patient_id.map(pseudo)
meta['image_key'] = meta.image_id.map(lambda s: pseudo(s, 20))

# Table de liaison (sensible) : séparée, utilisée uniquement pour retrouver le fichier
link_table = meta[['image_key', 'image_id']].copy()
link_table.to_parquet(f'{OUT_DIR}/_link_table_RESTRICTED.parquet', index=False)

meta = meta.drop(columns=['orig_w', 'orig_h', 'spacing_x', 'spacing_y'])
assert meta.patient_key.nunique() == meta.patient_id.nunique(), 'collision de hash'
print('Patients :', meta.patient_key.nunique())

In [ ]:
# Contrôle : pas de texte incrusté / PHI visible ? Inspection visuelle d'un échantillon
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, iid in zip(axes.ravel(), meta.image_id.sample(8, random_state=SEED)):
    if iid in path_map:
        ax.imshow(cv2.imread(path_map[iid], cv2.IMREAD_GRAYSCALE), cmap='gray')
    ax.set_title(meta.loc[meta.image_id == iid, 'labels'].values[0][:30], fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()

## 4. Alignement fiable image ↔ dossier patient

In [ ]:
meta['path'] = meta.image_id.map(path_map)
orphan_records = meta[meta.path.isna()]
orphan_images = set(path_map) - set(meta.image_id)
print(f'Dossiers sans image : {len(orphan_records)} | Images sans dossier : {len(orphan_images)}')
meta = meta[meta.path.notna()].copy()

# Le nom de fichier encode l'ID patient et le n° de suivi : on vérifie la cohérence
fname_pid = meta.image_id.str[:8].astype(int)
fname_fu  = meta.image_id.str[9:12].astype(int)
mismatch = (fname_pid != meta.patient_id) | (fname_fu != meta.follow_up)
print('Incohérences nom de fichier / dossier :', int(mismatch.sum()))
meta = meta[~mismatch]

# Images illisibles (contrôle sur échantillon pour le temps de calcul)
def readable(p):
    im = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    return im is not None and im.size > 0
chk = meta.sample(min(2000, len(meta)), random_state=SEED)
bad = chk[~chk.path.map(readable)]
print('Images illisibles (échantillon de 2000) :', len(bad))

In [ ]:
# Cohérence intra-patient : le sexe doit être constant, l'âge croissant avec le suivi
sex_incons = meta.groupby('patient_id').sex.nunique()
print('Patients avec sexe incohérent :', int((sex_incons > 1).sum()))
# Règle : on garde le sexe majoritaire du patient
maj_sex = meta.groupby('patient_id').sex.agg(lambda s: s.mode().iat[0])
meta['sex'] = meta.patient_id.map(maj_sex)

meta = meta.sort_values(['patient_id', 'follow_up'])
age_drop = meta.groupby('patient_id').age.diff() < -1
print('Âges décroissants avec le suivi :', int(age_drop.sum()))

## 5. Nettoyage

In [ ]:
# Âges aberrants (ex. 412 ans) -> manquants, ils seront imputés plus tard
meta['age'] = meta.age.astype(float)
meta.loc[(meta.age > 100) | (meta.age <= 0), 'age'] = np.nan
# Safe Harbor : âges > 89 plafonnés
meta['age'] = meta.age.clip(upper=90)

meta = meta.drop_duplicates(subset='image_id')
meta['sex'] = meta.sex.str.upper().str.strip()
meta['view'] = meta.view.str.upper().str.strip()
meta['labels'] = meta.labels.str.strip()
if SAMPLE_FRAC < 1:
    keep = meta.patient_id.drop_duplicates().sample(frac=SAMPLE_FRAC, random_state=SEED)
    meta = meta[meta.patient_id.isin(keep)]
print(meta.shape); meta.isna().sum()

## 6. Analyse exploratoire

In [ ]:
PATHOLOGIES = sorted({l for s in meta.labels for l in s.split('|')} - {'No Finding'})
print(len(PATHOLOGIES), PATHOLOGIES)
counts = pd.Series({p: meta.labels.str.contains(p, regex=False).sum() for p in PATHOLOGIES + ['No Finding']})
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
counts.sort_values().plot.barh(ax=axes[0], color='steelblue'); axes[0].set_title('Fréquence des labels (déséquilibre)')
sns.histplot(meta.age, bins=40, ax=axes[1]); axes[1].set_title('Âge')
meta.view.value_counts().plot.bar(ax=axes[2]); axes[2].set_title('Position de vue')
plt.show()

In [ ]:
Y = np.stack([meta.labels.str.contains(p, regex=False).values for p in PATHOLOGIES], 1).astype(int)
co = pd.DataFrame(Y.T @ Y, index=PATHOLOGIES, columns=PATHOLOGIES)
plt.figure(figsize=(10, 8)); sns.heatmap(np.log1p(co), cmap='mako'); plt.title('Co-occurrence (log)'); plt.show()

# Lien données cliniques <-> pathologies (justifie la fusion multimodale)
tmp = pd.DataFrame(Y, columns=PATHOLOGIES).assign(age=meta.age.values, view=meta.view.values, sex=meta.sex.values)
display(tmp.groupby('view')[PATHOLOGIES].mean().T.style.format('{:.3f}').background_gradient(axis=1))

## 7. Feature engineering

In [ ]:
# Labels multi-hot
for i, p in enumerate(PATHOLOGIES):
    meta[f'y_{p}'] = Y[:, i]
meta['y_any'] = (Y.sum(1) > 0).astype(int)
LABEL_COLS = [f'y_{p}' for p in PATHOLOGIES]

# Historique patient (uniquement le passé -> pas de fuite temporelle)
meta = meta.sort_values(['patient_id', 'follow_up']).reset_index(drop=True)
g = meta.groupby('patient_id')
meta['n_prior_exams'] = g.cumcount()
meta['n_total_exams'] = g.image_id.transform('count')
prior = g[LABEL_COLS].cumsum() - meta[LABEL_COLS]          # pathologies vues aux examens précédents
meta['n_prior_findings'] = prior.sum(1)
meta['had_prior_finding'] = (meta.n_prior_findings > 0).astype(int)
meta['is_followup'] = (meta.follow_up > 0).astype(int)
meta['age_group'] = pd.cut(meta.age, [0, 18, 40, 60, 80, 120], labels=['<18', '18-40', '40-60', '60-80', '80+'])
meta[['n_prior_exams', 'n_total_exams', 'n_prior_findings', 'had_prior_finding', 'age_group']].describe(include='all')

## 8. Valeurs manquantes
NIH contient peu de manquants (âges aberrants mis à NaN). Pour reproduire un contexte clinique réel (examens de labo non réalisés),
on ajoute des **indicateurs de manquants** et l'imputation est **ajustée uniquement sur le train** (dans le pipeline sklearn, étape 10).

In [ ]:
import missingno as msno
NUM_COLS = ['age', 'follow_up', 'n_prior_exams', 'n_total_exams', 'n_prior_findings']
CAT_COLS = ['sex', 'view']
BIN_COLS = ['had_prior_finding', 'is_followup']
meta['age_missing'] = meta.age.isna().astype(int)
BIN_COLS.append('age_missing')
msno.bar(meta[NUM_COLS + CAT_COLS], figsize=(8, 3)); plt.show()
print('Taux de manquants âge : %.4f%%' % (100 * meta.age.isna().mean()))

## 9. Split train / val / test par patient (anti-fuite)

In [ ]:
# Test = liste officielle NIH (déjà séparée par patient) ; val extraite du train_val par patient
test_df = meta[meta.image_id.isin(official_test)].copy()
trval = meta[meta.image_id.isin(official_trainval)].copy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.125, random_state=SEED)
tr_idx, va_idx = next(gss.split(trval, groups=trval.patient_id))
train_df, val_df = trval.iloc[tr_idx].copy(), trval.iloc[va_idx].copy()

for a, b, n in [(train_df, val_df, 'train/val'), (train_df, test_df, 'train/test'), (val_df, test_df, 'val/test')]:
    inter = set(a.patient_id) & set(b.patient_id)
    assert not inter, f'Fuite patient {n}: {len(inter)}'
print('Aucune fuite patient ✔')
split_stats = pd.DataFrame({n: d[LABEL_COLS].mean() for n, d in [('train', train_df), ('val', val_df), ('test', test_df)]})
print({n: len(d) for n, d in [('train', train_df), ('val', val_df), ('test', test_df)]})
split_stats.style.format('{:.3f}')

## 10a. Prétraitement tabulaire (ajusté sur le train seulement)

In [ ]:
tab_pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), NUM_COLS),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('oh', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), CAT_COLS),
    ('bin', 'passthrough', BIN_COLS),
])
tab_pre.fit(train_df)
TAB_FEATURES = list(tab_pre.get_feature_names_out())
X_tab = {n: tab_pre.transform(d).astype('float32') for n, d in [('train', train_df), ('val', val_df), ('test', test_df)]}
print(len(TAB_FEATURES), TAB_FEATURES)
joblib.dump(tab_pre, f'{OUT_DIR}/tabular_preprocessor.joblib')

# Poids de classes pour la BCE pondérée (déséquilibre)
pos = train_df[LABEL_COLS].mean().values
pos_weight = ((1 - pos) / np.clip(pos, 1e-6, None)).astype('float32')
pd.Series(pos_weight, index=PATHOLOGIES).round(1)

## 10b. Prétraitement image : CLAHE, redimensionnement, normalisation, augmentation

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
def load_xray(path, size=IMG_SIZE):
    im = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    im = clahe.apply(im)                                   # rehaussement local du contraste
    im = cv2.resize(im, (size, size), interpolation=cv2.INTER_AREA)
    return np.repeat(im[..., None], 3, axis=2)              # 3 canaux pour backbones ImageNet

NORM = A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
train_tf = A.Compose([
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, border_mode=cv2.BORDER_CONSTANT, p=0.7),
    A.RandomBrightnessContrast(0.1, 0.1, p=0.5),
    # pas de flip horizontal : le cœur est à gauche, l'anatomie n'est pas symétrique
    NORM, ToTensorV2()])
eval_tf = A.Compose([NORM, ToTensorV2()])

ex = train_df.sample(1, random_state=SEED).iloc[0]
raw = cv2.imread(ex.path, cv2.IMREAD_GRAYSCALE)
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(raw, cmap='gray'); ax[0].set_title('Originale')
ax[1].imshow(load_xray(ex.path)[..., 0], cmap='gray'); ax[1].set_title('CLAHE + 224')
aug = train_tf(image=load_xray(ex.path))['image'].permute(1, 2, 0).numpy()
ax[2].imshow((aug - aug.min()) / (aug.max() - aug.min())); ax[2].set_title('Augmentée')
[a.axis('off') for a in ax]; plt.show()

## 11. Dataset & DataLoader PyTorch multimodal

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class ChestXrayMultimodal(Dataset):
    def __init__(self, df, X_tab, tf):
        self.paths = df.path.values
        self.tab = torch.from_numpy(X_tab)
        self.y = torch.from_numpy(df[LABEL_COLS].values.astype('float32'))
        self.tf = tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = self.tf(image=load_xray(self.paths[i]))['image']
        return {'image': img, 'tabular': self.tab[i], 'label': self.y[i]}

loaders = {
    n: DataLoader(ChestXrayMultimodal(d, X_tab[n], train_tf if n == 'train' else eval_tf),
                  batch_size=32, shuffle=(n == 'train'), num_workers=2, pin_memory=True)
    for n, d in [('train', train_df), ('val', val_df), ('test', test_df)]}
b = next(iter(loaders['train']))
print({k: tuple(v.shape) for k, v in b.items()})

### Aperçu du modèle de fusion (tardive) alimenté par ce pipeline
Squelette prêt pour la phase d'entraînement ; la fusion par attention remplace `torch.cat` par une cross-attention.

In [ ]:
import torch.nn as nn, torchvision
class LateFusion(nn.Module):
    def __init__(self, n_tab, n_out):
        super().__init__()
        bb = torchvision.models.densenet121(weights='DEFAULT')
        self.img = nn.Sequential(bb.features, nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(1024, 256))
        self.tab = nn.Sequential(nn.Linear(n_tab, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 32), nn.ReLU())
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(256 + 32, n_out))
    def forward(self, image, tabular):
        return self.head(torch.cat([self.img(image), self.tab(tabular)], 1))

model = LateFusion(len(TAB_FEATURES), len(PATHOLOGIES))
with torch.no_grad(): print(model(b['image'][:2], b['tabular'][:2]).shape)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight))

## 12. Export des artefacts

In [ ]:
keep_cols = ['image_key', 'patient_key', 'age', 'sex', 'view', 'follow_up', 'n_prior_exams', 'n_total_exams',
             'n_prior_findings', 'had_prior_finding', 'is_followup', 'age_missing'] + LABEL_COLS
for n, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    d[keep_cols].to_parquet(f'{OUT_DIR}/{n}.parquet', index=False)   # sans patient_id ni nom de fichier
    np.save(f'{OUT_DIR}/X_tab_{n}.npy', X_tab[n])
json.dump({'pathologies': PATHOLOGIES, 'tab_features': TAB_FEATURES, 'pos_weight': pos_weight.tolist(),
           'img_size': IMG_SIZE, 'seed': SEED,
           'sizes': {n: len(d) for n, d in [('train', train_df), ('val', val_df), ('test', test_df)]}},
          open(f'{OUT_DIR}/config.json', 'w'), indent=2)
print(os.listdir(OUT_DIR))

## Récapitulatif
| Vigilance DataOps | Traitement appliqué |
|---|---|
| Anonymisation | ID patient haché SHA-256 salé, table de liaison isolée, âges > 89 plafonnés, métadonnées techniques supprimées |
| Valeurs manquantes | Âges aberrants → NaN, imputation médiane/mode ajustée sur train, indicateurs `*_missing` |
| Alignement image ↔ patient | Jointure par fichier, orphelins signalés, cohérence nom de fichier/ID/suivi, cohérence sexe intra-patient |
| Fuite de données | Split par patient + asserts, historique calculé sur le passé uniquement |

**Étape suivante :** entraîner les baselines (image seule, tabulaire seul) puis le modèle `LateFusion` et une variante à attention, évaluées en AUC-ROC par classe.